# 04 — Force resolution, time steps, and usable scales

We compare PM runs that start from exactly the same particles and phases. The experiment changes only the force mesh and the number of KDK steps.

The calculation is still a fixed-time teaching model. The finer run is an internal PM baseline, not an N-body truth simulation.

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path(os.path.abspath('.')).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

SEED = 2602
EVIDENCE_DOMAIN = "analytic-fixture"
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "red": "#C94C4C",
    "gray": "#626C78",
}
mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(0.99, 0.01, EVIDENCE_DOMAIN, ha="right", fontsize=7, color=COLORS["gray"])
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight")
    return path


import json

box_size = 500.0              # h^-1 Mpc
particle_side = 16
analysis_mesh = 32
final_time = 1.6              # toy time coordinate
baseline = (40, 36)           # force mesh, KDK steps
trials = [(16, 5), (16, 20), (32, 5), (32, 20)]

## The two diagnostics

$$
T(k)=\sqrt{\frac{P_{\rm run}(k)}{P_{\rm base}(k)}},
\qquad
r(k)=\frac{P_{\rm run,base}(k)}
{\sqrt{P_{\rm run}(k)P_{\rm base}(k)}}.
$$

$T$ measures amplitude; $r$ measures whether paired Fourier modes remain aligned. We accept a contiguous low-$k$ range while $|T-1|\leq0.05$ and $r\geq0.99$. A Nyquist frequency is only a sampling limit, not an accuracy claim.

In [ ]:
def cic_stencil_3d(positions, nmesh, box_size):
    mesh_position = (positions % box_size) / (box_size / nmesh)
    lower = np.floor(mesh_position).astype(int)
    fraction = mesh_position - lower
    for offset in np.ndindex(2, 2, 2):
        offset = np.asarray(offset)
        index = (lower + offset) % nmesh
        weight = np.prod(np.where(offset, fraction, 1 - fraction), axis=1)
        yield index[:, 0], index[:, 1], index[:, 2], weight


def cic_deposit_3d(positions, nmesh, box_size):
    mesh = np.zeros((nmesh, nmesh, nmesh))
    for ix, iy, iz, weight in cic_stencil_3d(positions, nmesh, box_size):
        np.add.at(mesh, (ix, iy, iz), weight)
    return mesh


def cic_interpolate_3d(vector_grid, positions, box_size):
    values = np.zeros((len(positions), 3))
    for ix, iy, iz, weight in cic_stencil_3d(
        positions, vector_grid.shape[0], box_size
    ):
        values += weight[:, None] * vector_grid[ix, iy, iz]
    return values


def pm_acceleration_3d(positions, nmesh, box_size):
    mass = cic_deposit_3d(positions, nmesh, box_size)
    delta = mass / mass.mean() - 1
    delta_k = np.fft.fftn(delta)
    k = 2 * np.pi * np.fft.fftfreq(nmesh, d=box_size / nmesh)
    kx, ky, kz = np.meshgrid(k, k, k, indexing="ij")
    k2 = kx**2 + ky**2 + kz**2
    phi_k = np.zeros_like(delta_k)
    nonzero = k2 > 0
    phi_k[nonzero] = -delta_k[nonzero] / k2[nonzero]

    k_grad = k.copy()
    k_grad[nmesh // 2] = 0
    gx, gy, gz = np.meshgrid(k_grad, k_grad, k_grad, indexing="ij")
    force_grid = np.stack([
        np.fft.ifftn(-1j * component * phi_k).real
        for component in (gx, gy, gz)
    ], axis=-1)
    return cic_interpolate_3d(force_grid, positions, box_size)


def evolve_mini_pm(positions, velocities, nmesh, nsteps, final_time, box_size):
    positions, velocities = positions.copy(), velocities.copy()
    dt = final_time / nsteps
    acceleration = pm_acceleration_3d(positions, nmesh, box_size)
    for _ in range(nsteps):
        velocity_half = velocities + 0.5 * dt * acceleration
        positions = (positions + dt * velocity_half) % box_size
        acceleration_new = pm_acceleration_3d(positions, nmesh, box_size)
        velocities = velocity_half + 0.5 * dt * acceleration_new
        acceleration = acceleration_new
    return positions


def make_initial_state(nside, box_size, seed):
    axis = (np.arange(nside) + 0.5) * box_size / nside
    q = np.stack(np.meshgrid(axis, axis, axis, indexing="ij"), axis=-1)
    rng = np.random.default_rng(seed)
    noise_k = np.fft.fftn(rng.normal(size=(nside, nside, nside)))
    k = 2 * np.pi * np.fft.fftfreq(nside, d=box_size / nside)
    kx, ky, kz = np.meshgrid(k, k, k, indexing="ij")
    k2 = kx**2 + ky**2 + kz**2
    smooth = noise_k * np.exp(-(np.sqrt(k2) / (0.7 * np.pi * nside / box_size))**4)

    displacement = []
    for component in (kx, ky, kz):
        component_k = np.zeros_like(smooth)
        nonzero = k2 > 0
        component_k[nonzero] = 1j * component[nonzero] * smooth[nonzero] / k2[nonzero]
        displacement.append(np.fft.ifftn(component_k).real)
    displacement = np.stack(displacement, axis=-1)
    displacement *= 0.035 * box_size / np.sqrt(np.mean(displacement**2))
    positions = (q + displacement) % box_size
    return positions.reshape(-1, 3), (0.45 * displacement).reshape(-1, 3)


positions_initial, velocities_initial = make_initial_state(
    particle_side, box_size, SEED + 40
)

In [ ]:
def isotropic_fidelity(delta, reference, box_size, edges):
    """Return shell-averaged T(k), r(k), and mode counts."""
    delta_k = np.fft.fftn(delta)
    reference_k = np.fft.fftn(reference)
    nmesh = delta.shape[0]
    k = 2 * np.pi * np.fft.fftfreq(nmesh, d=box_size / nmesh)
    kx, ky, kz = np.meshgrid(k, k, k, indexing="ij")
    kmag = np.sqrt(kx**2 + ky**2 + kz**2)

    shell = np.digitize(kmag.ravel(), edges) - 1
    valid = (shell >= 0) & (shell < len(edges) - 1) & (kmag.ravel() > 0)
    counts = np.bincount(shell[valid], minlength=len(edges) - 1)

    def shell_mean(values):
        total = np.bincount(
            shell[valid], weights=values.ravel()[valid], minlength=len(edges) - 1
        )
        return np.divide(
            total, counts, out=np.full_like(total, np.nan), where=counts > 0
        )

    power = shell_mean(np.abs(delta_k) ** 2)
    power_reference = shell_mean(np.abs(reference_k) ** 2)
    power_cross = shell_mean(np.real(delta_k * np.conj(reference_k)))

    # TODO 1: calculate T(k) and r(k) from these three spectra.
    transfer = np.full_like(power, np.nan)
    correlation = np.full_like(power, np.nan)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, transfer, correlation, counts


def contiguous_kmax(k, transfer, correlation, counts):
    # TODO 2: starting at the first supported bin, stop at the first failure of
    # |T-1| <= 0.05 or r >= 0.99. Return the last passing k.
    raise NotImplementedError

## A controlled $2\times2$ comparison

The four runs let us change steps at fixed mesh and mesh at fixed steps. Their relative-work proxy is

$$
W\propto (N_{\rm step}+1)N_{\rm mesh}^3\log_2N_{\rm mesh}.
$$

This is an FFT-dominated scaling estimate, not a hardware benchmark.

In [ ]:
baseline_positions = evolve_mini_pm(
    positions_initial, velocities_initial, *baseline, final_time, box_size
)
baseline_mass = cic_deposit_3d(baseline_positions, analysis_mesh, box_size)
baseline_delta = baseline_mass / baseline_mass.mean() - 1

k_nyquist = np.pi * analysis_mesh / box_size
edges = np.linspace(0, 0.85 * k_nyquist, 11)
results, curves = [], {}
for force_mesh, steps in trials:
    positions = evolve_mini_pm(
        positions_initial, velocities_initial,
        force_mesh, steps, final_time, box_size
    )
    mass = cic_deposit_3d(positions, analysis_mesh, box_size)
    delta = mass / mass.mean() - 1
    k, transfer, correlation, counts = isotropic_fidelity(
        delta, baseline_delta, box_size, edges
    )
    kmax = contiguous_kmax(k, transfer, correlation, counts)
    work = (steps + 1) * force_mesh**3 * np.log2(force_mesh)
    results.append({
        "force_mesh": force_mesh,
        "steps": steps,
        "work": float(work),
        "kmax": float(kmax),
    })
    curves[(force_mesh, steps)] = (k, transfer, correlation, counts)

minimum_work = min(result["work"] for result in results)
for result in results:
    result["relative_work"] = result.pop("work") / minimum_work

print("mesh  steps  relative work  kmax [h Mpc^-1]")
for result in results:
    print(
        f"{result['force_mesh']:>4}  {result['steps']:>5}"
        f"  {result['relative_work']:>13.2f}  {result['kmax']:>17.4f}"
    )

budget = {
    "schema_id": "icts26-day2-accuracy-v2",
    "evidence_domain": EVIDENCE_DOMAIN,
    "reference": {
        "force_mesh": baseline[0],
        "steps": baseline[1],
        "meaning": "finer internal PM baseline; not N-body truth",
    },
    "accuracy_rule": {"abs(T-1)": 0.05, "minimum_r": 0.99, "minimum_modes": 20},
    "work_proxy": "(steps+1) * force_mesh^3 * log2(force_mesh)",
    "results": results,
}
(OUTPUT_DIR / "04_accuracy_budget.json").write_text(json.dumps(budget, indent=2))

In [ ]:
colors = {16: COLORS["orange"], 32: COLORS["blue"]}
styles = {5: "--", 20: "-"}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)

axes[0].axhspan(0.95, 1.05, color=COLORS["green"], alpha=0.12)
axes[1].axhspan(0.99, 1.00, color=COLORS["green"], alpha=0.12)
for (mesh, steps), (k, transfer, correlation, counts) in curves.items():
    supported = counts >= 20
    label = f"{mesh} mesh, {steps} steps"
    axes[0].plot(k[supported], transfer[supported], "o",
                 ls=styles[steps], color=colors[mesh], label=label)
    axes[1].plot(k[supported], correlation[supported], "o",
                 ls=styles[steps], color=colors[mesh], label=label)

axes[0].set(title="amplitude", xlabel=r"$k\;[h\,{\rm Mpc}^{-1}]$", ylabel=r"$T(k)$")
axes[1].set(title="mode alignment", xlabel=r"$k\;[h\,{\rm Mpc}^{-1}]$", ylabel=r"$r(k)$")
axes[0].legend(fontsize=8)

work = [result["relative_work"] for result in results]
kmax = [result["kmax"] for result in results]
axes[2].scatter(work, kmax, s=55, color=COLORS["purple"])
label_offsets = {
    (16, 5): (4, 4),
    (16, 20): (4, 16),
    (32, 5): (4, 4),
    (32, 20): (4, 4),
}
for result in results:
    axes[2].annotate(
        f"{result['force_mesh']}×{result['steps']}",
        (result["relative_work"], result["kmax"]),
        xytext=label_offsets[(result["force_mesh"], result["steps"])],
        textcoords="offset points", fontsize=8,
    )
axes[2].set(
    title="accuracy versus work",
    xlabel="relative work proxy",
    ylabel=r"$k_{\max}\;[h\,{\rm Mpc}^{-1}]$",
)
savefig(fig, "04_accuracy_diagnostics.png")
plt.show()

## Connect the scale cut to a target

A rough halo scale is the inverse Lagrangian radius,

$$
R_{\rm L}=\left(\frac{3M}{4\pi\bar\rho_m}\right)^{1/3},
\qquad k_{\rm halo}\sim R_{\rm L}^{-1}.
$$

This is only a guide: halo abundance and structure also depend on peaks, tides, particle mass, and the halo definition.

In [ ]:
rho_mean = 0.30 * 2.775e11  # (h^-1 Msun) / (h^-1 Mpc)^3
target_mass = np.array([1e13, 1e14])
radius = (3 * target_mass / (4 * np.pi * rho_mean)) ** (1 / 3)
for mass, value in zip(target_mass, 1 / radius):
    print(f"M={mass:.0e} h^-1 Msun -> 1/R_L={value:.3f} h Mpc^-1")

## What to report

1. What changed when only the number of steps increased?
2. What changed when only the force mesh increased?
3. Which least-work run meets your target scale?

Even a passing matter-field comparison does not certify a halo catalog. Production codes such as DISCO-DJ and FastPM also require cosmological evolution and validation against a higher-fidelity reference.